In [2]:
import sys
sys.executable

'/usr/local/opt/python@3.11/bin/python3.11'

In [3]:
import ast
import pandas as pd
import numpy as np
from pygit2 import Object, Repository, GIT_SORT_TIME
from pygit2 import init_repository, Patch
from colorama import Fore
from tqdm import tqdm
import swifter
from pandarallel import pandarallel

In [4]:
pandarallel.initialize(progress_bar=True)

INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [5]:
library = 'lablup/backend.ai'

In [6]:
# import all push data
df_push = pd.DataFrame()
commit_urls = []
for val in np.arange(0, 100, 1):
    if val < 10:
        val = "0" + str(val)
    try:
        df_part = pd.read_csv(f'data/github_clean/pushEvent0000000000{val}.csv', index_col = 0)
        df_part['partition'] = val
        df_push = pd.concat([df_push, df_part])
    except:
        print(f'data/github_clean/pushEvent0000000000{val}.csv not found')

In [7]:
df_library = df_push[df_push['repo_name'] == library]
df_library.loc[:,'commit_urls'] = df_library['commit_urls'].apply(lambda x: ast.literal_eval(x))
commit_urls = df_library['commit_urls'].explode().tolist()

In [10]:
commit_urls

['https://api.github.com/repos/lablup/backend.ai/commits/c86f4dfeff22249776420ae09f05b4bb42a1cc17',
 'https://api.github.com/repos/lablup/backend.ai/commits/bede3c289c0a2b3620d07e061d986f52898654d9',
 'https://api.github.com/repos/lablup/backend.ai/commits/1d3fb81cd14a6ad29076d5b68bf023da8d6ee9db',
 'https://api.github.com/repos/lablup/backend.ai/commits/65825b700f625d28b0aa2007de74c2525aaeac47',
 'https://api.github.com/repos/lablup/backend.ai/commits/e6bafb07336e62c6348f69ffc50e04400c6f1ad1',
 'https://api.github.com/repos/lablup/backend.ai/commits/721bed41b4b49666e27ec8d98c4df1e6c9d6d1b9',
 'https://api.github.com/repos/lablup/backend.ai/commits/fac985115603436d8e86a2e9833683f718565be7',
 'https://api.github.com/repos/lablup/backend.ai/commits/bca7f0302562ac70c36ad4fe358e1066450d47bc',
 'https://api.github.com/repos/lablup/backend.ai/commits/a04289219a866223fdd853d825f4dba586e5476b',
 'https://api.github.com/repos/lablup/backend.ai/commits/65547bb7f9e5522d9f2b8933ce1d1c05126453e0',


In [387]:
repo = Repository('../repos/backend.ai')
sum = 0
for commit in commit_urls:
    if type(commit) != float:
        sha = commit.split("/")[-1]
        commit_a = repo.get(sha)
        if type(commit_a) != type(None):
            sum += 1
print(f"{sum} out of {len(commit_urls)} commit SHAs can be found ({100*round(sum/len(commit_urls), 3)}%)")

21523 out of 25698 commit SHAs can be found (83.8%)


In [388]:
def createCommitGroup(push_before, push_head, commit_urls, push_size):
    if push_size == 0:
        return [[]]
    elif push_size == 1:
        return [[push_before, push_head]]
    else:
        avail_commits = len(commit_urls)
        lst_result = [[push_before, commit_urls[0].split("/")[-1]]]
        for i in range(avail_commits-1):
            lst_result.append([commit_urls[i].split("/")[-1], commit_urls[i+1].split("/")[-1]])
        return lst_result

In [389]:
df_library['commit_groups'] = \
    df_library.apply(lambda x: createCommitGroup(x['push_before'], x['push_head'], x['commit_urls'], x['push_size']), axis = 1)
df_commit_groups = df_library[['push_id', 'repo_id', 'repo_name', 'actor_id', 'actor_login', 
                               'org_id', 'org_login', 'push_size', 'push_before', 'push_head',
                               'commit_groups']].explode('commit_groups')

/var/folders/5s/dvrxt95949x1pm_sjxm85lj00000gn/T/ipykernel_39288/1534266405.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_library['commit_groups'] = \


In [390]:
def returnCommitStats(x):
    if len(x) < 2:
        return []
    commit_parent_sha = x[0]
    commit_head_sha = x[1]
    commit_parent = repo.get(commit_parent_sha)
    commit_head = repo.get(commit_head_sha)
    if type(commit_parent) != type(None) and type(commit_head) != type(None):
        diff = repo.diff(commit_parent, commit_head, context_lines=0, interhunk_lines=0)
        commit_sha = commit_head_sha
        commit_author_name = commit_head.author.name
        commit_author_email = commit_head.author.email
        committer_author_name = commit_head.committer.name
        committer_author_email = commit_head.committer.email
        commit_message = commit_head.message
        commit_additions = diff.stats.insertions
        commit_deletions = diff.stats.deletions
        commit_changes_total = commit_additions + commit_deletions
        commit_files_changed_count = diff.stats.files_changed
        commit_time = commit_head.commit_time
        commit_file_changes = []
        
        for obj in diff:
            if type(obj) == Patch:
                additions = 0
                deletions = 0
                for hunk in obj.hunks:
                  for line in hunk.lines:
                    # The new_lineno represents the new location of the line after the patch. If it's -1, the line has been deleted.
                    if line.new_lineno == -1: 
                        deletions += 1
                    # Similarly, if a line did not previously have a place in the file, it's been added fresh. 
                    if line.old_lineno == -1: 
                        additions += 1
                commit_file_changes.append({'file':obj.delta.new_file.path,
                                            'additions': additions,
                                            'deletions': deletions,
                                            'total': additions + deletions})
        return [commit_sha, commit_author_name, commit_author_email, committer_author_name, committer_author_email,
                commit_message, commit_additions, commit_deletions, commit_changes_total, commit_files_changed_count,
                commit_file_changes]
    return []

In [392]:
%%time
commit_data = df_commit_groups['commit_groups'].parallel_apply(returnCommitStats)

CPU times: user 1.54 s, sys: 1.02 s, total: 2.56 s
Wall time: 12min 53s


In [393]:
df_commit = pd.DataFrame(commit_data.tolist(),
                        columns = ['commit sha', 'commit author name', 'commit author email', 'committer name',
                                   'commmitter email', 'commit message', 'commit additions', 'commit deletions',
                                   'commit changes total', 'commit files changed count', 'commit file changes'])

In [394]:
df_commit_final = pd.concat([df_commit_groups.reset_index(drop = True), df_commit], axis = 1)
df_commit_final

,push_id,repo_id,repo_name,actor_id,actor_login,org_id,org_login,push_size,push_before,push_head,...,commit author name,commit author email,committer name,commmitter email,commit message,commit additions,commit deletions,commit changes total,commit files changed count,commit file changes
0,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None
1,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Joongi Kim,joongi@lablup.com,GitHub,noreply@github.com,setup: Remove dependency to psycopg (#702)\n\n...,204.0,296.0,500.0,21.0,"[{'file': 'changes/702.breaking.md', 'addition..."
2,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Sanghun Lee,oper6909@gmail.com,GitHub,noreply@github.com,feat: Do not return AgentSummary if `hide-agen...,10.0,2.0,12.0,2.0,"[{'file': 'changes/699.feature.md', 'additions..."
3,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,Jonghyun Park,jpark@lablup.com,GitHub,noreply@github.com,feat: add `hide_agents` webserver config (#704...,6.0,1.0,7.0,4.0,"[{'file': 'changes/704.feature', 'additions': ..."
4,11550771535,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,84,f7288e25e5ba7e30ed58946942ea52413912d0bd,e27a7900fe25a6e392a3ff35da593d071422d65b,...,DaeHyun Sung,dhsung@lablup.com,GitHub,noreply@github.com,docs: update python default version (#724)\n\n...,1448.0,996.0,2444.0,11.0,"[{'file': 'docs/README.md', 'additions': 1, 'd..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25693,11646000079,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,286,6f1c5a2c203da153978b9b334709b33d6ba0383a,a2e85fd58c9d7fe10404bc26f6966f20c9097040,...,Jeongseok Kang,piono623@naver.com,GitHub,noreply@github.com,fix: Update integration test to work with the ...,96.0,55.0,151.0,8.0,"[{'file': 'changes/442.fix.md', 'additions': 1..."
25694,13838497595,70124554,lablup/backend.ai,2904494,kyujin-cho,11258248.0,lablup,1,437e4625f172057e645465c7084492cfeb0143e8,8e26fd42b6ec476b084f237b3b7a4a5c2f8e0c8f,...,Kyujin Cho,kyujin.cho@lablup.com,GitHub,noreply@github.com,feat: update POST `/service` API to also accep...,85.0,38.0,123.0,3.0,[{'file': 'src/ai/backend/client/cli/service.p...
25695,10610526310,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,1,37c8df3174c768b96f542e6511f33ffad219079e,b1c7115142f78f43743f85f417f1f7c852116dc5,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None
25696,10753912953,70124554,lablup/backend.ai,555156,achimnol,11258248.0,lablup,1,6eb9396d72c19b8388a1dec5f9749192f59c1f97,cc5fc1d3b750ad1cae16d18d01544efcc2453caf,...,None,None,None,None,None,NaN,NaN,NaN,NaN,None


In [418]:
df_repos

,github repo,count
0,dagster-io/dagster,43538
1,Azure/azure-sdk-for-python,30160
2,DataDog/dd-trace-py,26048
3,pulumi/pulumi,21919
4,ray-project/ray,20780
...,...,...
5846,hellock/cvbase,1
5847,heroku/heroku.py,1
5848,DenisCarriere/geocoder,1
5849,nickoala/telepot,1


In [417]:
df_repos = pd.DataFrame(df_push[['repo_name']].value_counts()).reset_index().rename(
    {'repo_name': 'github repo'}, axis = 1)
df_repos.to_csv('data/inputs/github_repo_grab_commits.csv')

In [430]:
import subprocess
lib = 'dagster-io/dagster.git'
lib_p2 = lib.split("/")[1]
lib_ren = lib.replace("/","___")
subprocess.Popen(["git", "clone", f"https://github.com/{lib}", f"{lib_ren}"], cwd = "../repos").wait()

Cloning into 'dagster-io___dagster.git'...
Updating files: 100% (7050/7050), done.


0